In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_05/data'
print(os.listdir(data_dir))

['views.csv', 'subscriptions.csv', 'catalog.csv', 'users.csv']


In [2]:
import pandas as pd
views = pd.read_csv(f'{data_dir}/views.csv', nrows=5)
subs = pd.read_csv(f'{data_dir}/subscriptions.csv', nrows=5)
catalog = pd.read_csv(f'{data_dir}/catalog.csv', nrows=5)
print("Views:", views.columns.tolist())
print("Subs:", subs.columns.tolist())
print("Catalog:", catalog.columns.tolist())

Views: ['view_id', 'view_date', 'user_id', 'title_id', 'watch_min', 'device']
Subs: ['user_id', 'sub_start', 'sub_end', 'plan']
Catalog: ['title_id', 'genre', 'release_year', 'runtime_min', 'is_series']


In [3]:
views = pd.read_csv(f'{data_dir}/views.csv')
subs = pd.read_csv(f'{data_dir}/subscriptions.csv')
catalog = pd.read_csv(f'{data_dir}/catalog.csv')

views['view_date'] = pd.to_datetime(views['view_date'])
subs['sub_start'] = pd.to_datetime(subs['sub_start'])
subs['sub_end'] = pd.to_datetime(subs['sub_end'])

# Merge views with subs
df = views.merge(subs, on='user_id', how='inner')

# Filter: view events that occurred while the user's subscription was active
df = df[(df['view_date'] >= df['sub_start']) & (df['view_date'] <= df['sub_end'])]

# Filter: within the final 7 days before sub_end
# "final 7 days before sub_end" means view_date >= sub_end - 7 days and view_date <= sub_end
df = df[(df['view_date'] >= df['sub_end'] - pd.Timedelta(days=7))]

# Merge with catalog to get genre
df = df.merge(catalog[['title_id', 'genre']], on='title_id', how='left')

# For each date, compute Documentary watch time as a share of total watch time from those events.
# Restrict to dates with at least 50 total watch minutes in this pre-end window.
daily_stats = df.groupby('view_date').apply(
    lambda x: pd.Series({
        'total_watch_min': x['watch_min'].sum(),
        'doc_watch_min': x.loc[x['genre'] == 'Documentary', 'watch_min'].sum()
    })
).reset_index()

daily_stats = daily_stats[daily_stats['total_watch_min'] >= 50]
daily_stats['doc_share'] = daily_stats['doc_watch_min'] / daily_stats['total_watch_min']

# On which date was the Documentary share highest? If there is a tie, use the earliest date.
daily_stats = daily_stats.sort_values(by=['doc_share', 'view_date'], ascending=[False, True])
print(daily_stats.head())


    view_date  total_watch_min  doc_watch_min  doc_share
6  2024-03-02               68             29   0.426471
20 2024-03-28              116             38   0.327586
10 2024-03-06               66             17   0.257576
21 2024-03-29               74             16   0.216216
18 2024-03-26              175             34   0.194286


In [4]:
df_test = views.merge(subs, on='user_id', how='inner')
df_test = df_test[(df_test['view_date'] >= df_test['sub_start']) & (df_test['view_date'] <= df_test['sub_end'])]
df_test['days_to_end'] = (df_test['sub_end'] - df_test['view_date']).dt.days
print(df_test['days_to_end'].value_counts().sort_index().head(15))

days_to_end
1      8
2      7
3      8
4      8
5     10
6      7
7      8
8      9
9      9
10    10
11     7
12     9
13     8
14     8
15     9
Name: count, dtype: int64


In [5]:
# "within the final 7 days before sub_end"
# If sub_end is 2024-03-10, the final 7 days are 2024-03-04 to 2024-03-10.
# This means days_to_end is between 0 and 6, or 1 and 7?
# "final 7 days before sub_end" usually means sub_end - 7 days < view_date <= sub_end
# Let's check if there are any days_to_end == 0
print(df_test[df_test['days_to_end'] == 0])


Empty DataFrame
Columns: [view_id, view_date, user_id, title_id, watch_min, device, sub_start, sub_end, plan, days_to_end]
Index: []


In [6]:
# So days_to_end is 1 to 7.
# Let's re-run the logic with days_to_end <= 7
df_test = df_test[df_test['days_to_end'] <= 7]
df_test = df_test.merge(catalog[['title_id', 'genre']], on='title_id', how='left')

daily_stats = df_test.groupby('view_date').apply(
    lambda x: pd.Series({
        'total_watch_min': x['watch_min'].sum(),
        'doc_watch_min': x.loc[x['genre'] == 'Documentary', 'watch_min'].sum()
    })
).reset_index()

daily_stats = daily_stats[daily_stats['total_watch_min'] >= 50]
daily_stats['doc_share'] = daily_stats['doc_watch_min'] / daily_stats['total_watch_min']

daily_stats = daily_stats.sort_values(by=['doc_share', 'view_date'], ascending=[False, True])
print(daily_stats.head())


    view_date  total_watch_min  doc_watch_min  doc_share
6  2024-03-02               68             29   0.426471
20 2024-03-28              116             38   0.327586
10 2024-03-06               66             17   0.257576
21 2024-03-29               74             16   0.216216
18 2024-03-26              175             34   0.194286


In [7]:
# Wait, what if days_to_end is 0?
# If sub_end is 2024-03-10, the final 7 days are 2024-03-04, 05, 06, 07, 08, 09, 10.
# 10 - 4 = 6.
# 10 - 10 = 0.
# So days_to_end should be <= 6?
# Let's check the exact wording: "within the final 7 days before sub_end"
# If sub_end is 2024-03-10, the 7 days before sub_end are 2024-03-03 to 2024-03-09?
# Or does it include sub_end? "before sub_end" could mean strictly before.
# Let's check if there are any views on sub_end.
print(views.merge(subs, on='user_id', how='inner').query('view_date == sub_end'))


Empty DataFrame
Columns: [view_id, view_date, user_id, title_id, watch_min, device, sub_start, sub_end, plan]
Index: []


In [8]:
# There are no views on sub_end.
# So days_to_end is 1 to 7.
# The code in cell 5 gives 2024-03-02 as the date with the highest doc_share.
# Let's double check the logic.
# "Focus only on view events that occurred while the user's subscription was active and within the final 7 days before sub_end."
# "For each date, compute Documentary watch time as a share of total watch time from those events."
# "Restrict to dates with at least 50 total watch minutes in this pre-end window."
# "On which date was the Documentary share highest? If there is a tie, use the earliest date."

# Let's print the top 1 result from cell 5.
print(daily_stats.iloc[0]['view_date'].strftime('%Y-%m-%d'))


2024-03-02
